# 05 - Model comparison

Fair side-by-side comparison of the trained models on the **identical** held-out test rows: headline metrics, ROC / PR / calibration curves, and a Brier-gated winner selection (identical rule to `src.scoring.best_model`). A separate panel compares the hazard model (08) against the static models per snapshot.

In [1]:
# =============================================================================
# §0 Setup
# =============================================================================
from __future__ import annotations
import sys, json, glob
from pathlib import Path

_here = Path.cwd().resolve()
while not (_here / "pyproject.toml").exists():
    _here = _here.parent
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import numpy as np
import pandas as pd
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             roc_curve, precision_recall_curve)
from sklearn.calibration import calibration_curve

from src import data_dir, tables_dir
from src import plotting as P
P.apply_plotly_theme()

TBL = tables_dir() / "05_comparison"
TBL.mkdir(parents=True, exist_ok=True)
# Brier tolerance for the calibration gate in model selection (see §5).
# Kept identical to src.scoring.BRIER_TOL so the notebook and serving agree.
BRIER_TOL = 0.005
print("Setup ok. Project:", _here.name)

Setup ok. Project: OverbookingAnalyse


In [ ]:
# =============================================================================
# §1 Discover and load every prediction parquet
# =============================================================================
# Convention from the model notebooks: Data/{NN}_{name}_predictions.parquet,
# each with at least the columns temporal_split, y_true, y_prob. New models are
# picked up automatically once trained.
PRED_GLOB = str(data_dir() / "*_predictions.parquet")
pred_paths = sorted(glob.glob(PRED_GLOB))

def _model_name(path: str) -> str:
    return Path(path).stem.replace("_predictions", "")

models: dict[str, pd.DataFrame] = {}
for p in pred_paths:
    df = pd.read_parquet(p)
    if "temporal_split" not in df.columns:
        print(f"  skipped {Path(p).name}: no temporal_split column"); continue
    test = df[df["temporal_split"] == "test"].copy()
    models[_model_name(p)] = test
    print(f"  {_model_name(p):14s}: {len(test):,} test rows")

assert models, (f"No predictions under {PRED_GLOB}. Train 01/02/03 first - "
                f"they write their *_predictions.parquet.")

# Fairness check: every model must be scored on the identical test rows.
_sizes = {m: (len(d), int(d["y_true"].sum())) for m, d in models.items()}
print("\n  (test rows, cancellations) per model:", _sizes)
if len(set(_sizes.values())) > 1:
    print("  WARNING: test slices differ across models - comparison only partly "
          "fair (did every model read the same split from 00?).")

  01_logreg     : 35,290 test rows
  02_xgboost    : 35,290 test rows
  03_histgb     : 35,290 test rows

  (test rows, cancellations) per model: {'01_logreg': (35290, 6163), '02_xgboost': (35290, 6163), '03_histgb': (35290, 6163)}


In [3]:
# =============================================================================
# §2 Headline metrics, including the base-rate (no-skill) floor
# =============================================================================
def lift_at(y_true, prob, frac):
    """Lift@k%: cancellation rate among the k% riskiest bookings / overall rate
    (>1 means better than random)."""
    n = max(1, int(len(prob) * frac))
    top = np.argsort(prob)[::-1][:n]
    return float(np.asarray(y_true)[top].mean() / np.asarray(y_true).mean())

def metrics_row(name, y_true, prob):
    return {"model": name,
            "auc":   roc_auc_score(y_true, prob),
            "ap":    average_precision_score(y_true, prob),
            "brier": brier_score_loss(y_true, prob),
            "lift@5%":  lift_at(y_true, prob, 0.05),
            "lift@10%": lift_at(y_true, prob, 0.10)}

_ref = next(iter(models.values()))
y_test = _ref["y_true"].astype(int).to_numpy()
base_rate = float(y_test.mean())

# Base-rate baseline: constant prediction p = base_rate. AUC = 0.5, AP ~ base_rate,
# Brier = base_rate*(1-base_rate). Every real model must beat this floor.
rows = [metrics_row("base_rate (floor)", y_test, np.full_like(y_test, base_rate, dtype=float))]
for name, d in models.items():
    rows.append(metrics_row(name, d["y_true"].astype(int).to_numpy(), d["y_prob"].to_numpy()))

comp = pd.DataFrame(rows).set_index("model").round(4)
comp.to_csv(TBL / "comparison.csv")
print(f"Base rate (test): {base_rate:.2%}\n")
print(comp.to_string())

Base rate (test): 17.46%

                      auc      ap   brier  lift@5%  lift@10%
model                                                       
base_rate (floor)  0.5000  0.1746  0.1441   0.0000    0.0000
01_logreg          0.7656  0.4147  0.1240   3.2299    2.6627
02_xgboost         0.7860  0.4472  0.1203   3.3402    2.9580
03_histgb          0.7868  0.4489  0.1201   3.3792    2.9353


In [ ]:
# =============================================================================
# §3 Grouped bar chart of the headline metrics
# =============================================================================
_metric_cols = ["auc", "ap", "brier"]
series = {m: [comp.loc[m, c] for c in _metric_cols] for m in comp.index}
fig_metrics = P.grouped_bars(_metric_cols, series,
                             title="Model comparison - headline metrics (test)",
                             yaxis_title="Value")
fig_metrics.show()
P.fig_to_html(fig_metrics, str(TBL / "metrics_bars.html"))

'/Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/reports/tables/05_comparison/metrics_bars.html'

In [ ]:
# =============================================================================
# §4 ROC, precision-recall and calibration curves
# =============================================================================
roc_curves, pr_curves, cal_curves = {}, {}, {}
for name, d in models.items():
    yt = d["y_true"].astype(int).to_numpy()
    yp = d["y_prob"].to_numpy()
    fpr, tpr, _ = roc_curve(yt, yp)
    roc_curves[name] = (fpr, tpr, roc_auc_score(yt, yp))
    prec, rec, _ = precision_recall_curve(yt, yp)
    pr_curves[name] = (rec, prec, average_precision_score(yt, yp))
    frac_pos, mean_pred = calibration_curve(yt, yp, n_bins=10)
    cal_curves[name] = (mean_pred, frac_pos)

P.roc_curve_fig(roc_curves, title="ROC - all models (test)").show()
P.pr_curve_fig(pr_curves, base_rate=base_rate, title="Precision-Recall (test)").show()
P.calibration_fig(cal_curves, title="Calibration (test)").show()

In [ ]:
# =============================================================================
# §5 Select the winner: highest AP among well-calibrated models (Brier gate)
# =============================================================================
# Selection rule (identical to src.scoring.best_model):
#   1. Rank by Average Precision - the right ranking metric at ~18% prevalence.
#   2. Only consider models whose test Brier is within BRIER_TOL of the best
#      (lowest) Brier, so we never ship a sharp ranker with poor calibration -
#      the probabilities feed the overbooking decision directly.
# AP alone would decide on differences far inside fold noise; the Brier gate
# makes the tie-break principled and consistent with serving.
cands = comp.drop(index="base_rate (floor)")
best_brier = cands["brier"].min()
eligible = cands[cands["brier"] <= best_brier + BRIER_TOL]
winner = eligible["ap"].idxmax()

print(f"Calibration gate: Brier <= {best_brier:.4f} + {BRIER_TOL} "
      f"-> eligible: {list(eligible.index)}")
print(f"Winner by AP within the gate: {winner}  "
      f"(AP={cands.loc[winner,'ap']:.4f}, AUC={cands.loc[winner,'auc']:.4f}, "
      f"Brier={cands.loc[winner,'brier']:.4f})")
print(f"Margin over base rate: +{cands.loc[winner,'ap']-base_rate:.4f} AP\n")

# The operating threshold is NOT chosen on test - the model notebooks fix the
# F1-optimal threshold on validation and write it to the model card. Read it back.
_card_path = tables_dir() / winner / "model_card.json"
if _card_path.exists():
    card = json.load(open(_card_path))
    print("  Operating point (fixed on validation):")
    for op in card.get("operating_points", []):
        print(f"    {op}")
else:
    print(f"  no model card at {_card_path} - check the model notebook.")

summary = {"winner": winner,
           "selection_metric": "average_precision (Brier-gated)",
           "brier_tol": BRIER_TOL,
           "base_rate": base_rate,
           "metrics": comp.to_dict(orient="index")}
json.dump(summary, open(TBL / "comparison_summary.json", "w"), indent=2, default=str)
print(f"\n  saved: {TBL/'comparison.csv'} + comparison_summary.json")

Calibration gate: Brier <= 0.1201 + 0.005 -> eligible: ['01_logreg', '02_xgboost', '03_histgb']
Winner by AP within the gate: 03_histgb  (AP=0.4489, AUC=0.7868, Brier=0.1201)
Margin over base rate: +0.2743 AP

  Operating point (fixed on validation):
    {'name': 'f1_optimal', 'threshold': 0.2443, 'precision': 0.3938, 'recall': 0.5348}

  saved: /Users/ruby.grambauer/Documents/DEV/OverbookingAnalyse/reports/tables/05_comparison/comparison.csv + comparison_summary.json


In [ ]:
# =============================================================================
# §6 Hazard model vs static models - per snapshot
# =============================================================================
# The hazard model (08) predicts cancel-in-window at each daily snapshot, so its
# unit is a per-snapshot CONDITIONAL probability - a different positive rate and a
# different test population than the per-booking static models. It must NOT be
# added as a row to the headline table above. Instead we compare static vs hazard
# AT each snapshot, which is the fair view: the hazard edge should grow as the
# arrival date approaches.
_haz = tables_dir() / "08_hazard" / "hazard_vs_static.csv"
if _haz.exists():
    hz = pd.read_csv(_haz).sort_values("snapshot_d")
    fig_h = P.lines_by_x(
        hz["snapshot_d"].tolist(),
        {"static AUC": hz["auc_static"].tolist(),
         "hazard AUC": hz["auc_hazard"].tolist()},
        title="Static vs hazard AUC per snapshot (days before arrival)",
        xaxis_title="Snapshot: days before arrival",
        yaxis_title="AUC (per-snapshot, conditional)",
        markers=True)
    fig_h.update_xaxes(autorange="reversed")  # arrival approaches toward the right
    fig_h.show()
    P.fig_to_html(fig_h, str(TBL / "hazard_vs_static.html"))
    print("hazard vs static (per snapshot):")
    print(hz.to_string(index=False))
    print("\nNote: per-snapshot CONDITIONAL metrics - NOT comparable to the "
          "per-booking headline table; read alongside it, not merged into it.")
else:
    print(f"no hazard comparison yet at {_haz} - run 08_hazard.ipynb to populate it.")

hazard vs static (per snapshot):
 snapshot_d  auc_static  auc_hazard    delta
          1    0.605533    0.717489 0.111956
          3    0.681886    0.749836 0.067950
          7    0.738082    0.766522 0.028440
         14    0.728806    0.757109 0.028304
         30    0.748352    0.768447 0.020095
         60    0.733158    0.747519 0.014361
         90    0.668779    0.675786 0.007006

Note: per-snapshot CONDITIONAL metrics — NOT comparable to the per-booking headline table; read alongside it, not merged into it.


## Walk-forward comparison

Honest one-step-ahead metrics (mean over folds) read from each model's card after 01/02/03 have run their deployment fit. AP + Brier gate picks the served static model.

In [ ]:
import src.scoring as sc, pandas as pd
rows = []
for m in ["logreg", "xgboost", "histgb"]:
    try:
        wfa = sc.load_model_card(m).get("walk_forward", {})
        rows.append({"model": m, **{k: (wfa.get(k, {}) or {}).get("mean") for k in ["auc","ap","brier","cost"]}})
    except Exception as e:
        rows.append({"model": m, "error": str(e)})
display(pd.DataFrame(rows))
try:
    print("served static model (AP + Brier gate):", sc.best_model())
except Exception as e:
    print("best_model unavailable:", e)